In [ ]:
from google.colab import files
uploaded = files.upload()

Saving List of Registered Milk Plants.pdf to List of Registered Milk Plants (2).pdf


# Extraction

In [ ]:
%pip install pdfplumber
import pdfplumber
import pandas as pd
import re

PDF_PATH = "/content/List of Registered Milk Plants.pdf"
OUT_PATH = "/content/registered_milk_plants.csv"

COLUMNS = [
    "State",
    "District",
    "Sr_No",
    "Plant_Name_Address",
    "Sector",
    "Licensing_Capacity(Lac Litre)",
    "Registration_No",
    "Date_of_Registration",
    "Avg_Milk_Procurement(Lac Litre)",
    "Products",
]

HEADER_ROW_MARKERS = {"sr", "sr no", "name & address of the"}


def clean(cell):
    """Flatten a possibly-multiline pdfplumber cell into a tidy single string."""
    if cell is None:
        return ""
    text = cell.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_header_row(row):
    first = clean(row[0]).lower()
    second = clean(row[1]).lower()
    return first.startswith("sr") or second.startswith("name & address")


def is_footnote_row(row):
    """Trailing glossary rows: '*SMP Skimmed Milk Powder', '* NPC - ...', etc."""
    first = clean(row[0])
    second = clean(row[1])
    return first in ("*", "") and (
        second.startswith("*")
        or "National Productivity Council" in second
        or "Export Inspection Agency" in second
        or second.strip() in ("N.A. - Not Available",)
        or clean(row[4]).startswith("*")
    )


def is_state_row(row):
    """State header can appear in col0 ('KARNATAKA STATE') or col1."""
    label = clean(row[0]) or clean(row[1])
    return label.strip().upper().endswith("STATE")


def is_district_row(row):
    """A row that's ONLY a location label (all other cells blank)."""
    first = clean(row[0])
    second = clean(row[1])
    rest = [clean(c) for c in row[2:]]

    if is_state_row(row):
        return True

    if first != "" or second == "":
        return False
    if any(rest):
        return False
    # District labels are short, upper-case, no digits
    return second.isupper() and not any(ch.isdigit() for ch in second)


def extract():
    records = []
    current_state = "HARYANA"
    current_district = ""

    with pdfplumber.open(PDF_PATH) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            for table in tables:
                for row in table:
                    row = [c if c is not None else "" for c in row]
                    if len(row) < 8:
                        row = row + [""] * (8 - len(row))

                    if is_header_row(row):
                        continue

                    if is_footnote_row(row):
                        continue

                    if is_district_row(row):
                        if is_state_row(row):
                            label = clean(row[0]) or clean(row[1])
                            current_state = label.upper().replace("STATE", "").strip()
                            current_district = ""
                        else:
                            current_district = clean(row[1])
                        continue

                    # Skip fully blank rows
                    if not any(clean(c) for c in row):
                        continue

                    sr_no = clean(row[0]).rstrip(".")
                    if sr_no == "" and not clean(row[1]):
                        continue

                    records.append(
                        {
                            "State": current_state,
                            "District": current_district,
                            "Sr_No": sr_no,
                            "Plant_Name_Address": clean(row[1]),
                            "Sector": clean(row[2]),
                            "Licensing_Capacity(Lac Litre)": clean(row[3]),
                            "Registration_No": clean(row[4]),
                            "Date_of_Registration": clean(row[5]),
                            "Avg_Milk_Procurement(Lac Litre)": clean(row[6]),
                            "Products": clean(row[7]),
                        }
                    )

    df = pd.DataFrame(records, columns=COLUMNS)
    return df


if __name__ == "__main__":
    df = extract()
    df.to_csv(OUT_PATH, index=False)
    print(f"Extracted {len(df)} rows -> {OUT_PATH}")
    print(df.head(10).to_string())

Extracted 50 rows -> /content/registered_milk_plants.csv
     State   District Sr_No                                                                                Plant_Name_Address   Sector Licensing_Capacity(Lac Litre)  Registration_No Date_of_Registration Avg_Milk_Procurement(Lac Litre)                                                         Products
0  HARYANA     AMBALA     1   Chief Executive Officer, Distt. Ambala Coop. Milk Producers Union Ltd., Milk Plant, AMBALA CITY    Coop.                          0.70             2/94               3.8.94                            0.70                                Liquid Milk, Ghee, Paneer, S.F.M.
1  HARYANA     AMBALA     2                      M/s. Smriti Products (P) Ltd.,55th Mile Stone, Highway Panchkula Road, SAHA.  Private                          0.75          28/2000             2.2.2000                            0.75     SMP, Butter, Ghee, Cheese, Skimmed Milk, WMP, Dairy Whitener
2  HARYANA  PANCHKULA     1     Managing Di

In [ ]:
df.head()


,State,District,Sr_No,Plant_Name_Address,Sector,Licensing_Capacity(Lac Litre),Registration_No,Date_of_Registration,Avg_Milk_Procurement(Lac Litre),Products
0,HARYANA,AMBALA,1,"Chief Executive Officer, Distt. Ambala Coop. M...",Coop.,0.70,2/94,3.8.94,0.70,"Liquid Milk, Ghee, Paneer, S.F.M."
1,HARYANA,AMBALA,2,"M/s. Smriti Products (P) Ltd.,55th Mile Stone,...",Private,0.75,28/2000,2.2.2000,0.75,"SMP, Butter, Ghee, Cheese, Skimmed Milk, WMP, ..."
2,HARYANA,PANCHKULA,1,"Managing Director, Haryana Dairy Development c...",Coop.,4.65,98/R- MMPO/93,22.9.93,,
3,HARYANA,FARIDABAD,1,"M/s. Nanak Dairy Plant, Nanak Dairy road, HODAL",Private,0.30,10/94,12.12.94,,
4,HARYANA,FARIDABAD,2,"M/s. Kwality Dairy (India), Tehsil Palwal, Vil...",Private,0.75,11/95,26.5.95,0.50,"Ice-Cream, Milk Powder, SMP, White Butter, Ghe..."


In [ ]:
df.shape

(50, 10)

# Data Cleaning

In [ ]:
df = df.drop(columns=["Sr_No"])

In [ ]:
df.shape

(50, 9)

In [ ]:
from google.colab import files

#df.to_csv('/content/registered_milk_plants_final.csv', index=False)
#files.download('/content/registered_milk_plants_final.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   State                            50 non-null     object
 1   District                         50 non-null     object
 2   Plant_Name_Address               50 non-null     object
 3   Sector                           50 non-null     object
 4   Licensing_Capacity(Lac Litre)    50 non-null     object
 5   Registration_No                  50 non-null     object
 6   Date_of_Registration             50 non-null     object
 7   Avg_Milk_Procurement(Lac Litre)  50 non-null     object
 8   Products                         50 non-null     object
dtypes: object(9)
memory usage: 3.6+ KB


In [ ]:
import numpy as np

# --- 1. State, District, Sector, Products, Plant_Name_Address, Registration_No: string dtype ---
for col in ['State', 'District', 'Sector', 'Plant_Name_Address', 'Registration_No', 'Products']:
    df[col] = df[col].astype('string')

# --- 2. Licensing_Capacity(Lac Litre): float ---
df['Licensing_Capacity(Lac Litre)'] = pd.to_numeric(
    df['Licensing_Capacity(Lac Litre)'].astype(str).str.extract(r'([\d.]+)')[0],
    errors='coerce'
).astype(float)

# --- 3. Avg_Milk_Procurement(Lac Litre): normalize missing markers, then convert to float ---
df['Avg_Milk_Procurement(Lac Litre)'] = (
    df['Avg_Milk_Procurement(Lac Litre)']
    .replace({'N.A.': np.nan, '-': np.nan, '': np.nan})
    .astype(float)
)

# --- 5. Date_of_Registration ---
df['Date_of_Registration'] = pd.to_datetime(
    df['Date_of_Registration'], format='%d.%m.%y', errors='coerce'
).fillna(
    pd.to_datetime(df['Date_of_Registration'], format='%d.%m.%Y', errors='coerce')
)

print(df.dtypes)
print(df.info())

State                              string[python]
District                           string[python]
Plant_Name_Address                 string[python]
Sector                             string[python]
Licensing_Capacity(Lac Litre)             float64
Registration_No                    string[python]
Date_of_Registration               datetime64[ns]
Avg_Milk_Procurement(Lac Litre)           float64
Products                           string[python]
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   State                            50 non-null     string        
 1   District                         50 non-null     string        
 2   Plant_Name_Address               50 non-null     string        
 3   Sector                           50 non-null     string        
 4   Licensing_Capac

In [ ]:
df.head()

,State,District,Plant_Name_Address,Sector,Licensing_Capacity(Lac Litre),Registration_No,Date_of_Registration,Avg_Milk_Procurement(Lac Litre),Products
0,HARYANA,AMBALA,"Chief Executive Officer, Distt. Ambala Coop. M...",Coop.,0.70,2/94,1994-08-03,0.70,"Liquid Milk, Ghee, Paneer, S.F.M."
1,HARYANA,AMBALA,"M/s. Smriti Products (P) Ltd.,55th Mile Stone,...",Private,0.75,28/2000,2000-02-02,0.75,"SMP, Butter, Ghee, Cheese, Skimmed Milk, WMP, ..."
2,HARYANA,PANCHKULA,"Managing Director, Haryana Dairy Development c...",Coop.,4.65,98/R- MMPO/93,1993-09-22,NaN,
3,HARYANA,FARIDABAD,"M/s. Nanak Dairy Plant, Nanak Dairy road, HODAL",Private,0.30,10/94,1994-12-12,NaN,
4,HARYANA,FARIDABAD,"M/s. Kwality Dairy (India), Tehsil Palwal, Vil...",Private,0.75,11/95,1995-05-26,0.50,"Ice-Cream, Milk Powder, SMP, White Butter, Ghe..."


In [ ]:
import numpy as np

# Convert empty strings to actual NaN first, then fill everything with "N/A"
df = df.replace(r'^\s*$', np.nan, regex=True)
df = df.fillna("N/A")

In [ ]:
df["District"].unique()

<StringArray>
[     'AMBALA',   'PANCHKULA',   'FARIDABAD',     'GURGAON',        'JIND',
     'JHAJJAR',      'KARNAL', 'KURUKSHETRA',       'MEWAT',     'PANIPAT',
      'REWARI',     'SONIPAT',       'SIRSA',      'ROHTAK',         'N/A']
Length: 15, dtype: string

In [ ]:
df["Sector"].unique()

<StringArray>
['Coop.', 'Private', 'GOI', 'N/A']
Length: 4, dtype: string

In [ ]:
df["Licensing_Capacity(Lac Litre)"].unique()

array([0.7, 0.75, 4.65, 0.3, 1.0, 3.5, 1.5, 0.4, 1500.0, 9000.0, 2.0,
       19.5, 0.15, 2.5, 0.6, 0.2, 0.22, 'N/A', 0.0, 2.2], dtype=object)

In [ ]:
df["Registration_No"].unique()

<StringArray>
[                    '2/94',                  '28/2000',
            '98/R- MMPO/93',                    '10/94',
                    '11/95',                    '27/99',
            '96/R- MMPO/93',           '112/R- MMPO/93',
          '149/R- MMPO/ 94',                  '33/2003',
                  '40/2007',                  '44/2010',
                  '47/2011',                     '3/94',
                    '18/95',                    '26/99',
           '145/R- MMPO/94',                     '5/94',
                  '38/2006',                  '29/2000',
            '74/R- MMPO/93',            '97/R- MMPO/93',
                  '43/2009',                  '45/2010',
                  '33/2002',           '176/R- MMPO/95',
           '101/R- MMPO/93',                     '1/94',
                    '23/96',                  '30/2000',
            '66/R- MMPO/93',            '78/R- MMPO/93',
                  '35/2004',                  '36/2004',
                 

In [ ]:
df["Date_of_Registration"].unique()

array([Timestamp('1994-08-03 00:00:00'), Timestamp('2000-02-02 00:00:00'),
       Timestamp('1993-09-22 00:00:00'), Timestamp('1994-12-12 00:00:00'),
       Timestamp('1995-05-26 00:00:00'), Timestamp('1999-11-26 00:00:00'),
       Timestamp('1993-10-15 00:00:00'), Timestamp('1994-10-12 00:00:00'),
       Timestamp('2003-07-25 00:00:00'), Timestamp('2007-09-07 00:00:00'),
       Timestamp('2010-03-18 00:00:00'), Timestamp('2011-02-04 00:00:00'),
       Timestamp('1995-12-08 00:00:00'), Timestamp('1999-06-22 00:00:00'),
       Timestamp('1994-06-27 00:00:00'), Timestamp('1994-12-05 00:00:00'),
       Timestamp('2006-09-25 00:00:00'), Timestamp('2000-03-01 00:00:00'),
       Timestamp('2005-12-21 00:00:00'), Timestamp('2009-12-28 00:00:00'),
       'N/A', Timestamp('2002-02-11 00:00:00'),
       Timestamp('1995-04-27 00:00:00'), Timestamp('1996-12-13 00:00:00'),
       Timestamp('2000-06-21 00:00:00'), Timestamp('1993-08-13 00:00:00'),
       Timestamp('1993-09-16 00:00:00'), Timestamp('

In [ ]:
df["Avg_Milk_Procurement(Lac Litre)"].unique()

array([0.7, 0.75, 'N/A', 0.5, 1.5, 1.0, 1.12, 0.19, 0.15, 1.25, 0.41,
       0.46, 1.03, 0.03, 0.16, 1.34, 0.3, 0.91], dtype=object)

In [ ]:
# Drop unwanted columns
df = df.drop(columns=["Plant_Name_Address", "Products"])

# Keep only the year from Date_of_Registration
df["Date_of_Registration"] = pd.to_datetime(
    df["Date_of_Registration"], dayfirst=True, errors="coerce"
).dt.year

df.head()

,State,District,Sector,Licensing_Capacity(Lac Litre),Registration_No,Date_of_Registration,Avg_Milk_Procurement(Lac Litre)
0,HARYANA,AMBALA,Coop.,0.7,2/94,1994.0,0.7
1,HARYANA,AMBALA,Private,0.75,28/2000,2000.0,0.75
2,HARYANA,PANCHKULA,Coop.,4.65,98/R- MMPO/93,1993.0,N/A
3,HARYANA,FARIDABAD,Private,0.3,10/94,1994.0,N/A
4,HARYANA,FARIDABAD,Private,0.75,11/95,1995.0,0.5


In [ ]:
df["Date_of_Registration"] = df["Date_of_Registration"].astype("Int64")

In [ ]:
df

,State,District,Sector,Licensing_Capacity(Lac Litre),Registration_No,Date_of_Registration,Avg_Milk_Procurement(Lac Litre)
0,HARYANA,AMBALA,Coop.,0.7,2/94,1994,0.7
1,HARYANA,AMBALA,Private,0.75,28/2000,2000,0.75
2,HARYANA,PANCHKULA,Coop.,4.65,98/R- MMPO/93,1993,N/A
3,HARYANA,FARIDABAD,Private,0.3,10/94,1994,N/A
4,HARYANA,FARIDABAD,Private,0.75,11/95,1995,0.5
5,HARYANA,FARIDABAD,Private,0.7,27/99,1999,N/A
6,HARYANA,FARIDABAD,Coop.,1.0,96/R- MMPO/93,1993,1.5
7,HARYANA,FARIDABAD,Private,1.0,112/R- MMPO/93,1993,1.0
8,HARYANA,FARIDABAD,Private,3.5,149/R- MMPO/ 94,1994,N/A
9,HARYANA,FARIDABAD,Private,1.5,33/2003,2003,1.12


# Feature Engineering

In [ ]:
df["Capacity_Utilization(%) "] = (
    pd.to_numeric(df["Avg_Milk_Procurement(Lac Litre)"], errors='coerce')
    /
    pd.to_numeric(df["Licensing_Capacity(Lac Litre)"], errors='coerce')
    * 100
).round(1)

In [ ]:
df.head()

,State,District,Sector,Licensing_Capacity(Lac Litre),Registration_No,Date_of_Registration,Avg_Milk_Procurement(Lac Litre),Capacity_Utilization(%)
0,HARYANA,AMBALA,Coop.,0.7,2/94,1994,0.7,100.0
1,HARYANA,AMBALA,Private,0.75,28/2000,2000,0.75,100.0
2,HARYANA,PANCHKULA,Coop.,4.65,98/R- MMPO/93,1993,N/A,NaN
3,HARYANA,FARIDABAD,Private,0.3,10/94,1994,N/A,NaN
4,HARYANA,FARIDABAD,Private,0.75,11/95,1995,0.5,66.7


In [ ]:
df["Capacity_Utilization(%) "] = df["Capacity_Utilization(%) "].fillna("N/A")

In [ ]:
df

,State,District,Sector,Licensing_Capacity(Lac Litre),Registration_No,Date_of_Registration,Avg_Milk_Procurement(Lac Litre),Capacity_Utilization(%)
0,HARYANA,AMBALA,Coop.,0.7,2/94,1994,0.7,100.0
1,HARYANA,AMBALA,Private,0.75,28/2000,2000,0.75,100.0
2,HARYANA,PANCHKULA,Coop.,4.65,98/R- MMPO/93,1993,N/A,N/A
3,HARYANA,FARIDABAD,Private,0.3,10/94,1994,N/A,N/A
4,HARYANA,FARIDABAD,Private,0.75,11/95,1995,0.5,66.7
5,HARYANA,FARIDABAD,Private,0.7,27/99,1999,N/A,N/A
6,HARYANA,FARIDABAD,Coop.,1.0,96/R- MMPO/93,1993,1.5,150.0
7,HARYANA,FARIDABAD,Private,1.0,112/R- MMPO/93,1993,1.0,100.0
8,HARYANA,FARIDABAD,Private,3.5,149/R- MMPO/ 94,1994,N/A,N/A
9,HARYANA,FARIDABAD,Private,1.5,33/2003,2003,1.12,74.7


In [ ]:
# Flag rows with mismatched units so they don't skew Lac Litre analysis
df["Capacity_Unit_Flag"] = df["Registration_No"].isin(["44/2010", "47/2011"]).map(
    {True: "MT Solids p.a.", False: "Lac Litre"}
)

# Optionally exclude them from capacity-based aggregations/charts
df_liquid_only = df[df["Capacity_Unit_Flag"] == "Lac Litre"]

In [ ]:
df

,State,District,Sector,Licensing_Capacity(Lac Litre),Registration_No,Date_of_Registration,Avg_Milk_Procurement(Lac Litre),Capacity_Utilization(%),Capacity_Unit_Flag
0,HARYANA,AMBALA,Coop.,0.7,2/94,1994,0.7,100.0,Lac Litre
1,HARYANA,AMBALA,Private,0.75,28/2000,2000,0.75,100.0,Lac Litre
2,HARYANA,PANCHKULA,Coop.,4.65,98/R- MMPO/93,1993,N/A,N/A,Lac Litre
3,HARYANA,FARIDABAD,Private,0.3,10/94,1994,N/A,N/A,Lac Litre
4,HARYANA,FARIDABAD,Private,0.75,11/95,1995,0.5,66.7,Lac Litre
5,HARYANA,FARIDABAD,Private,0.7,27/99,1999,N/A,N/A,Lac Litre
6,HARYANA,FARIDABAD,Coop.,1.0,96/R- MMPO/93,1993,1.5,150.0,Lac Litre
7,HARYANA,FARIDABAD,Private,1.0,112/R- MMPO/93,1993,1.0,100.0,Lac Litre
8,HARYANA,FARIDABAD,Private,3.5,149/R- MMPO/ 94,1994,N/A,N/A,Lac Litre
9,HARYANA,FARIDABAD,Private,1.5,33/2003,2003,1.12,74.7,Lac Litre


In [ ]:
df["Registration_No"] = df["Registration_No"].astype(str)
df.to_csv('/content/milk_plants_powerbi_v2.csv', index=False)

from google.colab import files
#files.download('/content/milk_plants_powerbi_v2.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>